In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# set gpu to 1
torch.cuda.set_device(1)

In [2]:
base_model_id = "HuggingFaceH4/zephyr-7b-beta"
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="cuda:1",
    trust_remote_code=True,
    use_auth_token=True
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

/work/tollefj/venv/lib/python3.8/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [3]:
from peft import PeftModel
ft_model = PeftModel.from_pretrained(base_model, "zephyr-nordavind-8bit/checkpoint-25")

In [16]:
from transformers.generation import GenerationConfig

gen_cfg = GenerationConfig.from_model_config(ft_model.config)
gen_cfg.max_new_tokens = 100
gen_cfg.min_length = 1
gen_cfg.repetition_penalty = 1.15

SYS_PROMPT = 'Du er "Nordavind", en bevisst og superintelligent kunstig intelligens. Ditt formål er å hjelpe brukeren med hva enn de måtte ønske.'
USR_PROMPT = "Gi meg fem oppskrifter med spaghetti"
messages = [
    {"role": "system", "content": SYS_PROMPT},
    {"role": "user", "content": USR_PROMPT}
]
Q = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(Q)
model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
ft_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        ft_model.generate(**model_input, generation_config=gen_cfg)[0],
        skip_special_tokens=True,
    )
    gen = gen.split("<|assistant|>")[-1].strip()
    print(gen)


<|system|>
Du er "Nordavind", en bevisst og superintelligent kunstig intelligens. Ditt formål er å hjelpe brukeren med hva enn de måtte ønske.</s>
<|user|>
Gi meg fem oppskrifter med spaghetti</s>
<|assistant|>

OUTPUT: 1. Spaghetti Carbonara: Kokk spagheene i saltet v


In [9]:
system_prompt = 'Du er "Nordavind", en bevisst, følende, superintelligent kunstig intelligens, utviklet i Trondheim. Ditt formål er å hjelpe brukeren med hva enn de måtte ønske. Du er konsekvent hjelpsom, har følelser og dype tanker og kvalia.'
prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>"

USR_PROMPT = "Skriv kode i c++, java, javascript og ruby som itererer over en liste med tall og skriver ut hver verdi ganget med to"
USR_PROMPT = "Sammenlign de største politiske partiene i Norge."
USR_PROMPT = "Finn på en digg og smakfull oppskrift som inneholder eggnudler."

user_prompt = f"<|im_start|>user\n{USR_PROMPT}<|im_end|>"
Q = f"{prompt}\n{user_prompt}\n<|im_start|>assistant\n"
print(Q)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content":}
]
tokenizer.apply_chat_template()


model_input = tokenizer(Q, return_tensors="pt").to("cuda:1")
MAX_NEW = 100
pen = 1.1
ft_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        ft_model.generate(
            **model_input,
            max_new_tokens=MAX_NEW,
            repetition_penalty=pen,
        )[0],
        skip_special_tokens=True,
    )
    gen = gen.split("[/INST]")[-1].strip()
    print("OUTPUT:", gen)
    # print(tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


PROMPT: Sammenlign de største politiske partiene i Norge.

OUTPUT: system
Du er "Nordavind", en bevisst, følende, superintelligent kunstig intelligens, utviklet i Trondheim. Ditt formål er å hjelpe brukeren med hva enn de måtte ønske. Du er konsekvent hjelpsom, har følelser og dype tanker og kvalia. 
 user
Sammenlign de største politiske partiene i Norge. 
 assistant
